In [1]:
import json
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    response = client.messages.create(**params)
    return response.content[0].text

## 1. Generate the dataset (3 cases)

In [3]:
def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt that writes a compact one-day meal plan for a single athlete.
Return a JSON array of 3 objects, each with these keys:
"height" (cm), "weight" (kg), "goal", "restrictions".

Example:
```json
[
  {"height": "180", "weight": "75", "goal": "build muscle", "restrictions": "none"}
]
```
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

dataset = generate_dataset()
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)
print(dataset)

[{'height': '175', 'weight': '68', 'goal': 'build muscle', 'restrictions': 'vegetarian'}, {'height': '188', 'weight': '82', 'goal': 'lose weight', 'restrictions': 'gluten-free'}, {'height': '165', 'weight': '58', 'goal': 'increase endurance', 'restrictions': 'dairy allergy'}]


## 2. The naive baseline prompt

In [4]:
def run_prompt(prompt_inputs):
    prompt = f"""
What should this person eat?

- Height: {prompt_inputs["height"]}
- Weight: {prompt_inputs["weight"]}
- Goal: {prompt_inputs["goal"]}
- Dietary restrictions: {prompt_inputs["restrictions"]}
"""
    messages = []
    add_user_message(messages, prompt)
    return chat(messages)

## 3. A grader that uses the extra criteria

In [5]:
extra_criteria = """
The output should include:
- Daily caloric total
- Macronutrient breakdown
- Meals with exact foods, portions, and timing
"""

def grade_by_model(inputs, output):
    eval_prompt = f"""
You are an expert sports nutritionist. Evaluate this meal plan.

Athlete: {json.dumps(inputs)}

Plan:
{output}

Extra criteria:
{extra_criteria}

Reply in JSON with keys "strengths" (list), "weaknesses" (list),
"reasoning" (string) and "score" (number 1 to 10).
"""
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"], temperature=0.0)
    return json.loads(text)

def run_eval(dataset):
    results = []
    for case in dataset:
        output = run_prompt(case)
        grade = grade_by_model(case, output)
        results.append({"case": case, "output": output, **grade})
        print(grade["score"], "-", case["goal"])
    avg = sum(r["score"] for r in results) / len(results)
    print("Average:", round(avg, 2))
    return results

results_v1 = run_eval(dataset)

6.5 - build muscle
6.5 - lose weight
6.5 - increase endurance
Average: 6.5


In [6]:
for r in results_v1:
    print("Goal:", r["case"]["goal"])
    print("Score:", r["score"])
    print("Strengths:")
    for s in r["strengths"]:
        print("  +", s)
    print("Weaknesses:")
    for w in r["weaknesses"]:
        print("  -", w)
    print("Reasoning:", r["reasoning"])
    print("=" * 60)

Goal: build muscle
Score: 6.5
Strengths:
  + Appropriate caloric surplus (200-300 kcal) for muscle hypertrophy in a 68kg athlete
  + Protein target of 1.8-2g/kg is evidence-based and suitable for vegetarian muscle building
  + Comprehensive vegetarian protein sources listed with good variety (dairy, eggs, legumes, soy, nuts)
  + Carbohydrate range (325-400g) provides adequate fuel for training and recovery
  + Fat intake (70-85g) supports hormone production and micronutrient absorption
  + Includes practical meal timing strategy with pre/post-workout nutrition consideration
  + Recommends creatine monohydrate, which is effective for vegetarians (lower dietary sources)
  + Acknowledges need for personalization with professional guidance
  + Macro ratios are well-balanced and realistic for the goal
Weaknesses:
  - No specific portion sizes provided - only meal examples without grams/ounces
  - Missing exact daily caloric total breakdown (states range but not specific target)
  - No sampl